# 03 Results tables

Tables 4, 5 and 6 and the cross-dataset summary, with confidence intervals, and LaTeX snippets written to `results/tables_tex/`. A table that does not exist yet is reported as missing, never filled in.

In [1]:
import json, os, sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd
from IPython.display import Image, display
pd.set_option("display.max_columns", 40, "display.width", 200, "display.precision", 4)
RESULTS, CHECKPOINTS = ROOT / "results", ROOT / "checkpoints"

def require(*paths, hint):
    missing = [str(p) for p in paths if not Path(p).exists()]
    if missing:
        print("MISSING:", *missing, sep="\n  ")
        print("To create:", hint)
    return not missing

def runs():
    return sorted(p for p in CHECKPOINTS.iterdir() if (p / "run.json").is_file()) if CHECKPOINTS.is_dir() else []


In [2]:
TEX = RESULTS / "tables_tex"; TEX.mkdir(parents=True, exist_ok=True)
def with_ci(table, value="value", lo="ci_lo", hi="ci_hi"):
    out = table.copy()
    out["value (95% CI)"] = [f"{v:.3f} ({a:.3f}-{b:.3f})" for v, a, b in zip(out[value], out[lo], out[hi])]
    return out

T4 = RESULTS / "table4_detection_grading_aptos_test.csv"
if require(T4, hint="python -m xaidr.cli evaluate --run <run id>"):
    t4 = pd.read_csv(T4)
    print("Model:", t4["model"].iloc[0]); print("Run:", t4["run_id"].iloc[0], "| threshold:", round(t4["detection_threshold"].iloc[0], 4))
    shown = with_ci(t4)[["task", "metric", "value (95% CI)", "n"]]
    display(shown); (TEX / "table4.tex").write_text(shown.to_latex(index=False))

Model: Fallback baseline, NOT the specified model (this study): convnext_tiny, 256 px
Run: 20261011-222646_convnext_tiny_oversample_seed42 | threshold: 0.7896


,task,metric,value (95% CI),n
0,detection,accuracy,0.989 (0.979-0.996),526
1,detection,sensitivity,0.984 (0.969-0.996),526
2,detection,specificity,0.993 (0.981-1.000),526
3,detection,auc,0.998 (0.995-1.000),526
4,grading,accuracy,0.867 (0.842-0.892),526
5,grading,macro_sensitivity,0.694 (0.641-0.749),526
6,grading,macro_auc,0.957 (0.946-0.969),526
7,grading,qwk,0.910 (0.890-0.929),526


In [3]:
T6 = RESULTS / "table6_comparative.csv"
if require(T6, hint="python -m xaidr.cli evaluate --run <run id>"):
    t6 = pd.read_csv(T6)
    display(t6[["method", "dataset", "grading_accuracy", "qwk", "source", "note"]])
    (TEX / "table6.tex").write_text(t6[["method", "dataset", "grading_accuracy", "qwk"]].to_latex(index=False, float_format="%.3f"))

,method,dataset,grading_accuracy,qwk,source,note
0,"Chilukoti et al. [48], 2024",APTOS 2019,NaN,0.9670,published,Checked against the source. EyePACS-pretrained...
1,"Shakibania et al. [27], 2024",APTOS 2019,0.8960,0.9300,published,Seen only as quoted by a second paper; the dra...
2,"Dinpajhouh & Seyyedsalehi [50], 2023",APTOS 2019,0.8369,0.8926,published,NOT VERIFIED: values copied from the draft; th...
3,"Fallback baseline, NOT the specified model (th...",APTOS 2019 (held-out 15% test split),0.8669,0.9096,"this build, run 20261011-222646_convnext_tiny_...",n = 526; one seed


In [4]:
T5 = RESULTS / "table5_xai_fidelity.csv"
if require(T5, hint="prepare IDRiD, then python -m xaidr.cli xai --run <run id>"):
    t5 = pd.read_csv(T5)
    primary = t5[t5["primary"]]
    cols = ["method", "lesion", "n_images", "mean_iou", "iou_ci_lo", "iou_ci_hi", "chance_iou", "max_iou",
            "mean_recall", "chance_recall", "pointing_acc", "chance_pointing", "p_holm", "exploratory"]
    display(primary[cols]); (TEX / "table5.tex").write_text(primary[cols].to_latex(index=False, float_format="%.3f"))

,method,lesion,n_images,mean_iou,iou_ci_lo,iou_ci_hi,chance_iou,max_iou,mean_recall,chance_recall,pointing_acc,chance_pointing,p_holm,exploratory
0,gradcam_pp,MA,81,0.0025,0.0020,0.0030,0.0014,0.0139,0.1989,0.1,0.0000,0.0014,1.3520e-07,False
1,gradcam_pp,HE,80,0.0196,0.0153,0.0243,0.0097,0.1124,0.2607,0.1,0.0250,0.0136,4.6779e-08,False
2,gradcam_pp,EX,81,0.0194,0.0145,0.0248,0.0092,0.1190,0.2821,0.1,0.0000,0.0119,1.0191e-03,False
3,gradcam_pp,any,81,0.0422,0.0362,0.0486,0.0198,0.2585,0.2754,0.1,0.0247,0.0292,6.8821e-09,False
12,shap,MA,81,0.0022,0.0018,0.0027,0.0014,0.0139,0.1824,0.1,0.0000,0.0014,3.5767e-06,False
13,shap,HE,80,0.0209,0.0151,0.0271,0.0097,0.1124,0.2246,0.1,0.0000,0.0136,3.1296e-07,False
14,shap,EX,81,0.0102,0.0077,0.0127,0.0092,0.1190,0.1786,0.1,0.0123,0.0119,2.3174e-01,False
15,shap,any,81,0.0333,0.0276,0.0396,0.0198,0.2585,0.2027,0.1,0.0247,0.0292,3.1296e-07,False
24,lime,MA,81,0.0020,0.0015,0.0024,0.0014,0.0139,0.1558,0.1,0.0000,0.0014,3.6245e-03,False
25,lime,HE,80,0.0198,0.0144,0.0256,0.0097,0.1124,0.2008,0.1,0.0500,0.0136,3.7984e-06,False


In [5]:
CD = RESULTS / "crossdataset_summary.csv"
if require(CD, hint="python -m xaidr.cli crossdataset --run <run id>"):
    cd = pd.read_csv(CD)
    display(cd[["dataset", "split", "n", "status"]].drop_duplicates())
    done = cd[cd["status"] == "computed"]
    display(with_ci(done)[["dataset", "task", "metric", "value (95% CI)", "n", "detection_definition"]])
    (TEX / "crossdataset.tex").write_text(with_ci(done)[["dataset", "task", "metric", "value (95% CI)", "n"]].to_latex(index=False))

,dataset,split,n,status
0,aptos2019,test,526,computed
8,idrid,external_idrid,516,computed
16,messidor2,external_messidor2,0,not available


,dataset,task,metric,value (95% CI),n,detection_definition
0,aptos2019,detection,accuracy,0.989 (0.979-0.996),526,any DR (grade >= 1)
1,aptos2019,detection,sensitivity,0.984 (0.969-0.996),526,any DR (grade >= 1)
2,aptos2019,detection,specificity,0.993 (0.981-1.000),526,any DR (grade >= 1)
3,aptos2019,detection,auc,0.998 (0.995-1.000),526,any DR (grade >= 1)
4,aptos2019,grading,accuracy,0.867 (0.842-0.892),526,any DR (grade >= 1)
5,aptos2019,grading,macro_sensitivity,0.694 (0.641-0.749),526,any DR (grade >= 1)
6,aptos2019,grading,macro_auc,0.957 (0.946-0.969),526,any DR (grade >= 1)
7,aptos2019,grading,qwk,0.910 (0.890-0.929),526,any DR (grade >= 1)
8,idrid,detection,accuracy,0.818 (0.789-0.845),516,any DR (grade >= 1)
9,idrid,detection,sensitivity,0.968 (0.948-0.986),516,any DR (grade >= 1)
